In [61]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import ast

In [42]:
#Parsing Geojson file of ZIP Codes
gdf_ZIP_PA = gpd.read_file('../Spatial Joins/ACLED Join/pa_pennsylvania_zip_codes_geo.min.json')

#Renaming df
gdf_neighbZIP_PA = gdf_ZIP_PA

In [43]:
#Checking CRS
gdf_neighbZIP_PA.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

#### Extracting adjacent ZIP Codes

In [45]:
#Creating new column to store bordering ZIP Codes
gdf_neighbZIP_PA['bordering_ZCTA'] = None

for index, zip_polygon in gdf_neighbZIP_PA.iterrows():

    #Iterating over each polygon to define neighbours (i.e. ZIP Codes with touching borders)
    #Method: touches() returns TRUE/FALSE statements for all polygons that share a common border
    neighbors = gdf_neighbZIP_PA[gdf_neighbZIP_PA.geometry.touches(zip_polygon.geometry)]['ZCTA5CE10'].tolist()

    #filters out current ZIP code so only distinct neighbours are inlucded
    neighbors = [str(neighbor) for neighbor in neighbors if str(neighbor) != str(zip_polygon['ZCTA5CE10'])]
    gdf_neighbZIP_PA.at[index, 'bordering_ZCTA'] = ', '.join(neighbors)

In [78]:
#Renaming ZIP column 
gdf_neighbZIP_PA.rename(columns={'ZCTA5CE10': 'ZCTA'}, inplace=True)

In [79]:
#print results
print(gdf_neighbZIP_PA[['ZCTA', 'bordering_ZCTA']])

       ZCTA                                     bordering_ZCTA
0     17301                [17362, 17316, 17364, 17350, 17331]
1     17353                [17325, 17320, 17222, 17307, 17343]
2     17325  [17353, 17320, 17307, 17343, 17304, 17350, 173...
3     15018         [15642, 15037, 15135, 15089, 15028, 15083]
4     15215                [15238, 15223, 15201, 15206, 15116]
...     ...                                                ...
1793  17329                              [17362, 17327, 17331]
1794  17363                       [17327, 17349, 17322, 17352]
1795  17318                              [17347, 17406, 17345]
1796  17361                              [17327, 17349, 17355]
1797  17355                              [17327, 17349, 17361]

[1798 rows x 2 columns]


In [67]:
#Convert to csv
gdf_neighbZIP_PA.to_csv("NeighbouringZCTAs.csv", index=False)

In [68]:
#Reading relevant datasets

#Loading conflict data with all ZIP Codes
Conflict_all_ZIP = pd.read_csv('../Dataset/conflict_all_ZIP.csv')

#Loading 
NB_ZCTA = pd.read_csv('../feature space/NeighbouringZCTAs.csv')

In [83]:
#Mapping ZIP Codes to conflict events by creating pivot table for conflict events
pivot_conflicts = Conflict_all_ZIP.pivot_table(index=['ZCTA', 'month'], values='event_count', aggfunc='sum')

In [70]:
#converting string representation of list to actual list
NB_ZCTA['bordering_ZCTA'] = NB_ZCTA['bordering_ZCTA'].apply(ast.literal_eval)

In [71]:
#Renaming df so the function does not overwrite the conflict_all_ZIP df 
Conflict_all_ZIP_nb = Conflict_all_ZIP.copy()

In [86]:
#Function to sum conflict events in all neighbouring ZIP codes

def sum_neighbor_conflicts(ZCTA, month):
    try:
        #Converting ZCTA to float (because pivot_conflicts has ZIP codes as floats)
        ZCTA = float(ZCTA)
        month = str(month) 
        
        #Getting list of neighboring ZIP codes 
        neighbors = NB_ZCTA.loc[NB_ZCTA['ZCTA'] == ZCTA, 'bordering_ZCTA'].values[0]

        #Ensuring this is a list of strings
        neighbors = ast.literal_eval(neighbors) if isinstance(neighbors, str) else neighbors

        #Summing nr of conflicts for all neighbours in the list
        total_conflicts = 0
        for neighbor in neighbors:
            neighbor_float = float(neighbor)
            if (neighbor_float, month) in pivot_conflicts.index:
                total_conflicts += pivot_conflicts.loc[(neighbor_float, month), 'event_count']
        return total_conflicts
    except Exception as e:

        #Error handling: If no neighbors or no conflict events available, return 0
        return 0
    

# Converting variables to int
Conflict_all_ZIP_nb['ZCTA'] = Conflict_all_ZIP_nb['ZCTA'].astype(int)
Conflict_all_ZIP_nb['fatalities'] = Conflict_all_ZIP_nb['fatalities'].astype(int)
Conflict_all_ZIP_nb['event_count'] = Conflict_all_ZIP_nb['event_count'].astype(int)
Conflict_all_ZIP_nb['neighbor_conflicts'] = Conflict_all_ZIP_nb['neighbor_conflicts'].astype(int)

# Applying function to each row in Conflict_all_ZIP_nb
Conflict_all_ZIP_nb['neighbor_conflicts'] = Conflict_all_ZIP_nb.apply(
    lambda row: sum_neighbor_conflicts(row['ZCTA'], row['month']), axis=1
)

print(Conflict_all_ZIP_nb.head(50))


     ZCTA    month  admin2  fatalities  event_count  neighbor_conflicts
0   15001  2022-01  Beaver           0            1                 0.0
1   15001  2022-02  Beaver           0            0                 0.0
2   15001  2022-03  Beaver           0            0                 0.0
3   15001  2022-04  Beaver           0            0                 0.0
4   15001  2022-05  Beaver           0            0                 0.0
5   15001  2022-06  Beaver           0            0                 0.0
6   15001  2022-07  Beaver           0            0                 0.0
7   15001  2022-08  Beaver           0            0                 0.0
8   15001  2022-09  Beaver           0            1                 0.0
9   15001  2022-10  Beaver           0            0                 1.0
10  15001  2022-11  Beaver           0            0                 0.0
11  15001  2022-12  Beaver           0            0                 1.0
12  15003  2022-01       0           0            0             

In [87]:
#Saving to csv
Conflict_all_ZIP_nb.to_csv('Conflict_all_ZIP_nb.csv', index=False)